In [1]:
print(" hello world")

 hello world


In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec
import itertools
import random
import pandas as pd
import numpy as np
from uuid import uuid4

ModuleNotFoundError: No module named 'dotenv'

In [ ]:
from pinecone import ServerlessSpec

index_name = "pinecone-data"

# Create only if it doesn't already exist
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

print("✅ Index created (or already exists)")

✅ Index created (or already exists)


In [ ]:
print(pc.list_indexes())

IndexList([<name='pinecone-data', dim=1536, ready=True>])


In [ ]:
index = pc.Index("pinecone-data")

print(index.describe_index_stats())

DescribeIndexStatsResponse(dimension=1536, total_vector_count=0, metric='cosine', namespaces=0)


In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
pinecone_api_key = os.getenv("PINECONE_API_KEY")

client = OpenAI(api_key=openai_api_key)
pc = Pinecone(api_key=pinecone_api_key)

In [ ]:
url = "https://rajpurkar.github.io/SQuAD-explorer/dataset/train-v2.0.json"

data = requests.get(url).json()

print(type(data))
print(data.keys())

<class 'dict'>
dict_keys(['version', 'data'])


In [ ]:
records = []

for article in data["data"]:
    title = article["title"]

    for para in article["paragraphs"]:
        context = para["context"]

        records.append({
            "text_id": str(uuid4()),
            "title": title,
            "text": context
        })

df = pd.DataFrame(records)

df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)

print("Total documents:", len(df))
df.head()

Total documents: 19029


,text_id,title,text
0,f383823d-751e-4114-921f-3f7c9a029d89,Beyoncé,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...
1,993b8dd9-27f6-437c-ab70-7d531bb00cdb,Beyoncé,Following the disbandment of Destiny's Child i...
2,d4374999-0444-491a-b83e-b2a910656c18,Beyoncé,"A self-described ""modern-day feminist"", Beyonc..."
3,f46c4fbc-b167-4572-9b95-67b8ff5873b6,Beyoncé,"Beyoncé Giselle Knowles was born in Houston, T..."
4,00bbf117-9327-41ee-95dc-b7220de70239,Beyoncé,Beyoncé attended St. Mary's Elementary School ...


In [ ]:
batch_size = 100

batches = [
    df[i:i + batch_size]
    for i in range(0, len(df), batch_size)
]

print(f"Total batches: {len(batches)}")
print(f"First batch size: {len(batches[0])}")
print(f"Last batch size: {len(batches[-1])}")

Total batches: 191
First batch size: 100
Last batch size: 29


In [ ]:
batch_size = 100

batches = [df[i:i+batch_size] for i in range(0, len(df), batch_size)]

In [ ]:
for batch in batches:

    ids = [str(uuid4()) for _ in range(len(batch))]

    texts = batch["text"].tolist()

    metadata = [
        {
            "title": row["title"],
            "text": row["text"]
        }
        for _, row in batch.iterrows()
    ]

In [ ]:
response = client.embeddings.create(
    input=texts,
    model="text-embedding-3-small"
)

vectors = [r.embedding for r in response.data]

In [ ]:
index.upsert(
    vectors=list(zip(ids, vectors, metadata)),
    namespace="squad_rag_dataset"
)

UpsertResponse(upserted_count=29)

In [ ]:
def retrieve_docs(query):

    query_embedding = client.embeddings.create(
        input=query,
        model="text-embedding-3-small"
    ).data[0].embedding

    results = index.query(
        vector=query_embedding,
        top_k=3,
        include_metadata=True,
        namespace="squad_rag_dataset"
    )

    retrieved_docs = []
    sources = []

    for match in results["matches"]:
        retrieved_docs.append(match["metadata"]["text"])
        sources.append(match["metadata"]["title"])

    return retrieved_docs, sources

In [ ]:
def build_prompt(query, docs):

    prompt_start = "Answer the question using the context below:\n\n"

    context = "\n\n".join(docs)

    prompt_end = f"\n\nQuestion:{query}"

    return prompt_start + context + prompt_end

In [ ]:
def question_answering(prompt, sources):

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant."
            },
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    answer = response.choices[0].message.content

    return {
        "answer": answer,
        "sources": sources
    }

In [ ]:
query = "What is machine learning?"

docs, sources = retrieve_docs(query)

prompt = build_prompt(query, docs)

result = question_answering(prompt, sources)

print("Answer:\n", result["answer"])
print("\nSources:", result["sources"])

Answer:
 The provided context does not contain any information about machine learning. However, I can provide a brief definition: 

Machine learning is a subset of artificial intelligence (AI) that involves the use of algorithms and statistical models to enable computers to perform specific tasks without using explicit instructions. Instead, they learn from and make predictions or decisions based on data. It involves training models on large datasets to recognize patterns and make inferences or predictions based on new data.

Sources: ['Matter', 'Matter', 'Matter']
